In [1]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

d:\Repos\Gen-AI\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Indexing (Document Ingestion)

In [2]:
video_id = "31qyMKNB2RA" # Gfr50f6ZBvo
try:
    ytt_api = YouTubeTranscriptApi()
    try:
        transcript_list = ytt_api.fetch(video_id, languages=["en"])

    except NoTranscriptFound:
        print("English not found, using Hindi...")
        transcript_list = ytt_api.fetch(video_id, languages=["hi"])
    
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print("Transcript of the video:\n", transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

English not found, using Hindi...
Transcript of the video:
 हाय गाइज़, माय नेम इज नितेश एंड यू आर वेलकम टू माय YouTube चैनल। इस वीडियो में भी हम लोग अपना एजेंटिक एआई यूजिंग लैंग ग्राफ प्लेलिस्ट कंटिन्यू करेंगे। और आज का वीडियो इस प्लेलिस्ट का थर्ड वीडियो होगा। इसके पहले मैंने इस प्लेलिस्ट में दो वीडियोस और ऐड किए हैं। अह जो पहला वीडियो था वहां पे मैंने आपको एजेंटिक एआई और जनरेटिव एआई के बीच में जो की डिफरेंसेस हैं वो बताए थे। अह जो सेकंड वीडियो था वहां पे हमने एक डिटेल्ड ओवरव्यू लिया था कि एजेंटिक एआई होता क्या है? मैंने वहां पे आपको डेफिनेशन बताया। मैंने इनफैक्ट एक प्रैक्टिकल सिनेरियो लेकर के आपको समझाया था कि एजेंटिक एआई कैसे किसी प्रॉब्लम को सॉल्व करता है। हमने वहां पर एक ऑटोमेटेड हायरिंग का एग्जांपल लिया था और मैंने आपको स्टेप बाय स्टेप विजुअली दिखाया था कि एजेंटिक एi उस प्रॉब्लम को सॉल्व करेगा। उस एग्जांपल को देखने के बाद मैंने आपको एजेंटिक एआई के की कैरेक्टरिस्टिक्स और ट्रेड्स भी बताए थे और लास्ट में मैंने आपको एजेंटिक एआई के की कंपोनेंट्स बताए थे। तो नट में आई कैन से कि अभी तक हम

### Indexing: Text Splitting

In [3]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

print(f"Number of chunks created: {len(chunks)}")

Number of chunks created: 92


### Indexing: Embedding & Storing

In [4]:
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2547.91it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Retrieval

In [5]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

### Augmentation

In [6]:
llm = ChatGroq(model="openai/gpt-oss-safeguard-20b", temperature=0)
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

# Generation

In [7]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

parser = StrOutputParser()

main_chain = parallel_chain | prompt | llm | parser

In [8]:
main_chain.invoke('List all the challenges mentioned in the video and explain them in detail.')

'**Challenges highlighted in the video**\n\n| # | Challenge | Why it matters | How it shows up in the example |\n|---|-----------|----------------|--------------------------------|\n| 1 | **Statelessness of LangChain** | LangChain is designed to chain together LLM calls and simple tools, but it does **not** keep any persistent “state” between those calls.  The only memory it offers is *conversational* – i.e., what the LLM has said in the current chat. | In the hiring‑request workflow you need to remember whether a JD was approved or not, and then decide whether to go to “post JD” or back to “create JD.”  LangChain can’t automatically keep that flag; you must handle it yourself. |\n| 2 | **No built‑in key‑value store** | While you can store the chat history, LangChain provides no mechanism to store arbitrary data (e.g., `{"jd_approved": True}`) and retrieve it later. | The video explains that you would have to create a dictionary at the top of your code and manually update it every time

### Improvements to make a chatbot production grade

1. UI based enhancements
2. Evaluation
    - Ragas
    - LangSmith

3. Indexing
    - Document Ingestion
    - Text Splitting
    - Vector Store
4. Retrieval
    1. Pre-Retrieval
        - Query rewriting using LLM
        - Multi-query generation
        - Domain aware routing

    2. During Retrieval
        1. MMR
        2. Hybrid Retrieval
        3. Reranking

    3. Post-Retrieval - Contextual Compression

5. Augmentation
    - Prompt Templating
    - Answer grounding
    - Context window optimization

6. Generation
    - Answer with Citation
    - Guard railing

7. System Design
    - Multimodal
    - Agentic
    - Memory based